In [1]:
import xarray as xr
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier


from sklearn.metrics import classification_report, r2_score
import torch
from torch.utils.data import TensorDataset, DataLoader
from skimage.metrics import structural_similarity as ssim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm
from sklearn.metrics import classification_report, jaccard_score, cohen_kappa_score, confusion_matrix
import seaborn as sns
from astral import moon

from io import BytesIO
from openpyxl.drawing.image import Image
import random

In [2]:
target_res = 0.125
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.5, -60.0, -44.875]
split_year = 2023
split_year_test = 2025
min_time, max_time = pd.to_datetime("2009-01-01"), pd.to_datetime("2025-12-31")
sp = "HKP"
todas = False

fishing_ds = xr.open_dataset(f"../data/processed/targets/cpue_{target_res}.nc").sel(FAOspp=sp)
fishing = fishing_ds["CPUE_class"]
fishing = fishing.fillna(0)

mask_ds = xr.open_dataset(f"../data/processed/static/area_pesca_{target_res}.nc")
mask = mask_ds["mask"]
mask = mask.fillna(0)
mask = mask.broadcast_like(fishing)

temp_ds = xr.open_dataset("../data/processed/dynamic/to_surface.nc")
temp_ds = temp_ds.rename({"to":"TO"})
temp = temp_ds["TO"]
# temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_bottom_ds = xr.open_dataset("../data/processed/dynamic/temp_bottom.nc")
temp_bottom_ds = temp_bottom_ds.rename({"to": "TOB"})
temp_bottom = temp_bottom_ds["TOB"]
# temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

chl_ds = xr.open_dataset("../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
# chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

mixed_ds = xr.open_dataset("../data/processed/dynamic/mixed_layer.nc")
mixed_ds = mixed_ds.rename({"mlotst":"MLOTST"})
mixed = mixed_ds["MLOTST"]
# mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

depth_ds = xr.open_dataset("../data/processed/static/depth.nc")
depth_ds = depth_ds.rename({"depth":"PROF"})
depth = depth_ds["PROF"]
# depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../data/processed/dynamic/zo_surface.nc")
zo_ds = zo_ds.rename({"zo":"ZO"})
zo = zo_ds["ZO"]
# zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

so_ds = xr.open_dataset("../data/processed/dynamic/so_surface.nc")
so_ds = so_ds.rename({"so":"SO"})
so = so_ds["SO"]
# so = (so - so.mean()) / so.std()
so = so.fillna(0)

ugo_ds = xr.open_dataset("../data/processed/dynamic/ugo_surface.nc")
ugo_ds = ugo_ds.rename({"ugo":"UGO"})
ugo = ugo_ds["UGO"]
# ugo = (ugo - ugo.mean()) / ugo.std()
ugo = ugo.fillna(0)

vgo_ds = xr.open_dataset("../data/processed/dynamic/vgo_surface.nc")
vgo_ds = vgo_ds.rename({"vgo":"VGO"})
vgo = vgo_ds["VGO"]
# vgo = (vgo - vgo.mean()) / vgo.std()
vgo = vgo.fillna(0)

pp = xr.open_dataset("../data/processed/dynamic/pp.nc")
pp = pp["PP"]
# pp = (pp - pp.mean()) / pp.std()
pp = pp.fillna(0)

cdm = xr.open_dataset("../data/processed/dynamic/cdm.nc")
cdm = cdm["CDM"]
# cdm = (cdm - cdm.mean()) / cdm.std()
cdm = cdm.fillna(0)

spm = xr.open_dataset("../data/processed/dynamic/spm.nc")
spm = spm["SPM"]
# spm = (spm - spm.mean()) / spm.std()    
spm = spm.fillna(0)

zsd = xr.open_dataset("../data/processed/dynamic/zsd.nc")
zsd = zsd["ZSD"]
# zsd = (zsd - zsd.mean()) / zsd.std()
zsd = zsd.fillna(0)


month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)
month_sin.name = "MSEN"
month_cos.name = "MCOS"
month = month.broadcast_like(temp)

times = pd.DatetimeIndex(temp.time.values)

moon_ = np.array([moon.phase(t) for t in times])
moon_phase = xr.DataArray(moon_, coords={"time": temp.time}, dims=["time"], name="FL")
# moon_phase = (moon_phase - moon_phase.mean()) / moon_phase.std()
moon_phase = moon_phase.fillna(0)
moon_phase = moon_phase.broadcast_like(temp)


year = temp["time"].dt.year
# year = (year - year.mean()) / year.std()
year = year.broadcast_like(temp)
year.name = "Año"

lat = temp["lat"]
lon = temp["lon"]
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp)
lat.name = "LAT"
lon.name = "LON"

temp, temp_bottom, chl, mixed, depth, month_sin, month_cos, lat, lon, zo, so, month, year, ugo, vgo, pp, cdm, spm, zsd, moon_phase = xr.align(
    temp, temp_bottom, chl, mixed, depth, month_sin, month_cos, lat, lon, zo, so, month, year, vgo, ugo, pp, cdm, spm, zsd, moon_phase, join="inner")

fishing, mask = xr.align(fishing, mask, join="inner")


#area de pesca y un poco alrededor
croppedT = lambda da: da.sel(
    lon=slice(min_lon-0.5, max_lon+0.5),
    lat=slice(min_lat-0.5, max_lat+0.5),
    time=slice(min_time, max_time)
)

# area global
cropped = lambda da: da.sel(
    lon=slice(min_lon-0, max_lon+2),
    lat=slice(min_lat-4, max_lat+1),
    time=slice(min_time, max_time)
)

fishing = croppedT(fishing)
mask = croppedT(mask)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)
month = cropped(month)
month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)
year = cropped(year)
ugo = cropped(ugo)
vgo = cropped(vgo)
pp = cropped(pp)
cdm = cropped(cdm)
spm = cropped(spm)
zsd = cropped(zsd)
moon_phase = cropped(moon_phase)

In [3]:
y = fishing
y = y.transpose("time", "lat", "lon")

m = mask 
m = m.transpose("time", "lat", "lon")



vars_ = [temp,
        ugo, vgo, mixed,
        cdm, chl, spm,
        month_sin, month_cos,
        ]

if todas:
    vars_ = [temp, 
        so,  zo, temp_bottom,
        ugo, vgo, mixed,
        cdm, chl,  pp, zsd, spm,
        month_sin, month_cos,
        year,
        moon_phase,
        lat, lon,
        depth,
        ]


vars_names = [v.name for v in vars_]
in_channels = len(vars_)

X = xr.concat(vars_, dim="channel")
X = X.transpose("time", "channel", "lat", "lon")

print(X.shape)




# -------- Data Standardization (Z-score Normalization) --------
X = X.assign_coords(channel=vars_names)
epsilon = 1e-8 

# 1. Isolate the training period ONLY to calculate mean/std (prevents data leakage)
train_slice = X.sel(time=slice(None, f"{split_year-1}-12-31"))
train_mean = train_slice.mean(dim=["time", "lat", "lon"], skipna=True)
train_std  = train_slice.std(dim=["time", "lat", "lon"], skipna=True)

# 2. Set mean=0 and std=1 for cyclical features
for feat in ["month_sin", "month_cos"]:
    if feat in train_mean.coords["channel"].values:
        train_mean.loc[dict(channel=feat)] = 0.0
        train_std.loc[dict(channel=feat)] = 1.0 - epsilon

# 3. Apply the transformation to the ENTIRE dataset
X_scaled = (X - train_mean) / (train_std + epsilon)

# -------- Window Creation --------
# We modify this slightly to also return the exact timestamp of the target (y)
def create_windows_full(X_ds, y_ds, m_ds, window=12):
    X_data = X_ds.values   # (time, channels, H, W)
    y_data = y_ds.values   # (time, H, W)
    m_data = m_ds.values   # (time, H, W)
    time_data = y_ds.time.values # Track the timestamps
    
    X_seq, y_seq, m_seq, time_seq = [], [], [], []

    for i in range(len(X_data) - window):
        X_seq.append(X_data[i:i+window]) # 12 months history
        y_seq.append(y_data[i+window])   # 1 month target
        m_seq.append(m_data[i+window])   # 1 month mask target
        time_seq.append(time_data[i+window]) # The timestamp of the target
        
    return (
        torch.tensor(np.stack(X_seq), dtype=torch.float32),
        torch.tensor(np.stack(y_seq), dtype=torch.float32),
        torch.tensor(np.stack(m_seq), dtype=torch.float32),
        pd.to_datetime(time_seq) # Convert to pandas datetime for easy filtering
    )

window = 12

# 4. Create windows over the entire scaled dataset
X_all, y_all, m_all, t_all = create_windows_full(X_scaled, y, m, window)


# -------- Split into Train/Val/Test --------
# 5. Split the tensors based on the target's year
train_mask = t_all.year < split_year
val_mask = (t_all.year >= split_year) & (t_all.year < split_year_test)
test_mask = t_all.year >= split_year_test

X_train, y_train, m_train = X_all[train_mask], y_all[train_mask], m_all[train_mask]
X_val, y_val, m_val = X_all[val_mask], y_all[val_mask], m_all[val_mask]
X_test, y_test, m_test = X_all[test_mask], y_all[test_mask], m_all[test_mask]

print("Train shape:", X_train.shape)  # Will be N - 12
print("Val shape:", X_val.shape)      # Will retain all 24 months
print("Test shape:", X_test.shape)    # Will retain all 24 months

###### Data Loaders ######
batch_size = 24

train_loader = DataLoader(
    TensorDataset(X_train, y_train, m_train),
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val, m_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test, m_test),
    batch_size=batch_size,
    shuffle=False
)


(204, 9, 62, 25)
Train shape: torch.Size([156, 12, 9, 62, 25])
Val shape: torch.Size([24, 12, 9, 62, 25])
Test shape: torch.Size([12, 12, 9, 62, 25])


In [4]:
def prep_ml_data(X, y, m=None):
    """
    Converts 5D input tensors and 3D target tensors into 2D arrays 
    compatible with scikit-learn Random Forest. Handles both Torch tensors and Numpy arrays.
    """
    # Move to CPU and convert to numpy if dealing with PyTorch Tensors
    if isinstance(X, torch.Tensor):
        X = X.detach().cpu().numpy()
    if isinstance(y, torch.Tensor):
        y = y.detach().cpu().numpy()
    if isinstance(m, torch.Tensor):
        m = m.detach().cpu().numpy()

    N = X.shape[0]
    
    # Flatten features: (N, T, C, H_x, W_x) -> (N, T * C * H_x * W_x)
    # For your train data: (72, 12, 11, 94, 48) -> (72, 595584)
    X_rf = X.reshape(N, -1)
    
    # Flatten targets: (N, H_y, W_y) -> (N, H_y * W_y)
    # For your train data: (72, 37, 20) -> (72, 740)
    y_rf = y.reshape(N, -1)
    
    if m is not None:
        m_rf = m.reshape(N, -1)
        return X_rf, y_rf, m_rf
    else:
        return X_rf, y_rf

X_train_ml, y_train_ml, m_train_ml = prep_ml_data(X_train, y_train, m_train)
X_val_ml, y_val_ml, m_val_ml = prep_ml_data(X_val, y_val, m_val)
X_test_ml, y_test_ml, m_test_ml = prep_ml_data(X_test, y_test, m_test)


H_val_test, W_val_test = y_val.shape[1], y_val.shape[2]
H_out_test, W_out_test = y_test.shape[1], y_test.shape[2]

m_true = m_test_ml.reshape(-1, H_out_test, W_out_test)
y_true = y_test_ml.reshape(-1, H_out_test, W_out_test)

m_true_val = m_val_ml.reshape(-1, H_val_test, W_val_test)
y_true_val = y_val_ml.reshape(-1, H_val_test, W_val_test)

mask_flat = m_test_ml.astype(bool)
mask_falt_val = m_val_ml.astype(bool)

In [5]:
def compute_metrics(y_test_ml, y_pred_flat, y_true, mask_spatial):

    # =====================================================
    # 1. FLATTEN FOR METRICS
    # =====================================================
    y_test_flat = y_test_ml.flatten()
    y_pred_flat = y_pred_flat.flatten()
    
    mask_spatial = mask_spatial.astype(bool)
    mask_flat = mask_spatial.flatten().astype(bool)
    

    y_test_masked = y_test_flat[mask_flat]
    y_pred_masked = y_pred_flat[mask_flat]
    # =====================================================
    # 2. Classification Metrics
    # =====================================================
    print("=== Classification Report ===")
    report = pd.DataFrame(classification_report(y_test_masked, y_pred_masked, labels=[0, 1, 2], output_dict=True)).T
    print(report)
    
    iou_per_class = jaccard_score(y_test_masked, y_pred_masked, average=None, labels=[0, 1, 2])
    kappa = cohen_kappa_score(y_test_masked, y_pred_masked)
    
    # print("=== Spatial Segmentation Metrics ===")
    # print(f"IoU Class 1 (Low CPUE):      {iou_per_class[1]:.3f}")
    # print(f"IoU Class 2 (High CPUE):     {iou_per_class[2]:.3f}")
    # print(f"Cohen's Kappa Score:         {kappa:.3f}\n")
    
    # New Plot: Confusion Matrix
    class_labels = ['Nada (0)', 'CPUE bajo (1)', 'CPUE alto (2)']
    cm = pd.DataFrame(confusion_matrix(y_test_masked, y_pred_masked, labels=[0, 1, 2]), index=class_labels, columns=class_labels)
    
    # =====================================================
    # SSIM
    # =====================================================
    H_out, W_out = y_true.shape[1], y_true.shape[2]
    y_pred3D = y_pred_flat.reshape(-1, H_out, W_out)
    mask_spatial = mask_spatial.astype(bool)

    data_range = (y_true[mask_spatial].max()-y_true[mask_spatial].min())

    ssim_scores = []

    for i in range(y_true.shape[0]):

        _, ssim_map = ssim(
            y_true[i],
            y_pred3D[i],
            data_range=data_range,
            win_size=5,
            full=True
        )

        valid_ssim_pixels = ssim_map[mask_spatial[i]]

        if len(valid_ssim_pixels) > 0:
            ssim_scores.append(np.mean(valid_ssim_pixels))

    mean_ssim = np.mean(ssim_scores)

    # print(f"Mean Masked SSIM: {mean_ssim:.4f}\n")

    # =====================================================
    # PLOTTING
    # =====================================================
    y_pred_mean = np.mean(y_pred3D, axis=0)
    y_true_mean = np.mean(y_true, axis=0)

    error = y_pred_mean - y_true_mean

    # plt.figure(figsize=(12, 4))

    # ax1 = plt.subplot(1, 3, 1)
    # plt.title("Mean Test Truth")
    # plt.imshow(y_true_mean, cmap='viridis')
    # plt.colorbar()
    # ax1.invert_yaxis()

    # ax2 = plt.subplot(1, 3, 2)
    # plt.title("Mean Test Prediction")
    # plt.imshow(y_pred_mean, cmap='viridis')
    # plt.colorbar()
    # ax2.invert_yaxis()

    # ax3 = plt.subplot(1, 3, 3)
    # plt.title("Mean Error (Pred - True)")
    # plt.imshow(error, cmap='coolwarm', norm=CenteredNorm())
    # plt.colorbar()
    # ax3.invert_yaxis()

    # plt.tight_layout()
    # plt.show()

    y_pred_plot2 = np.sum(y_pred3D, axis=(1, 2))
    y_true_plot2 = np.sum(y_true, axis=(1, 2))
    r2_plot2 = r2_score(y_true_plot2, y_pred_plot2)

    return report, iou_per_class, kappa, cm, y_true_plot2, y_pred_plot2, r2_plot2


In [6]:
iters = 10

# -------- Initialize Storage DataFrames/Lists for Iterations --------
importance_all_df = pd.DataFrame(columns=vars_names)
reports_list = []
iou_list = []
kappa_list = []
r2_plot_list = []

for i in range(iters):
    # Set dynamic random state per iteration just like the regression loop
    model = RandomForestClassifier(n_estimators=200, max_features="sqrt", n_jobs=-1, random_state=random.randint(0, 1000))
    model.fit(X_train_ml, y_train_ml)

    y_pred_flat = model.predict(X_test_ml)

    # Compute metrics
    report, iou_per_class, kappa, cm, y_true_plot, y_pred_plot, r2_plot = compute_metrics(y_test_ml, y_pred_flat, y_true, m_true)
    
    # Store scalar values
    kappa_list.append(kappa)
    r2_plot_list.append(r2_plot)
    
    # Store dataframes/arrays for structural averaging
    reports_list.append(report)
    iou_list.append(np.array(iou_per_class).flatten())

    # -------- Feature Importance Extraction --------
    raw_importances = model.feature_importances_
    T = X_train.shape[1]   # window (12)
    C = X_train.shape[2]   # number of variables/channels
    H_x = X_train.shape[3] # height (lat)
    W_x = X_train.shape[4] # width (lon)

    reshaped_importances = raw_importances.reshape(T, C, H_x, W_x)
    variable_importances = reshaped_importances.sum(axis=(0, 2, 3))
    
    importance_all_df.loc[len(importance_all_df)] = variable_importances


# ==============================================================================
# -------- POST-LOOP AGGREGATION & FORMATTING (Mean ± Std) --------
# ==============================================================================

# 1. Feature Importance Summary
importance_summary_df = pd.DataFrame({
    "Variable": vars_names, 
    "Importance": [f"{importance_all_df[var].mean():.3f}±{importance_all_df[var].std():.3f}" for var in vars_names],
    "Mean Importance": [importance_all_df[var].mean() for var in vars_names]
})
importance_summary_df = importance_summary_df.sort_values(by="Mean Importance", ascending=False).reset_index(drop=True)
importance_summary_df = importance_summary_df.drop(columns=["Mean Importance"])

# 2. Classification Report Summary (Averages the structural metrics table)
concat_reports = pd.concat(reports_list)
report_mean = concat_reports.groupby(concat_reports.index).mean()
report_std = concat_reports.groupby(concat_reports.index).std()

report_summary_df = pd.DataFrame(index=report_mean.index)
for col in report_mean.columns:
    report_summary_df[col] = [f"{m:.3f}±{s:.3f}" for m, s in zip(report_mean[col], report_std[col])]

# Reorder index to match the original structure of report if necessary
report_summary_df = report_summary_df.reindex(report.index)

# 3. IoU Per Class Summary
iou_arr = np.array(iou_list) # Shape: (iters, 3)
iou_mean = iou_arr.mean(axis=0)
iou_std = iou_arr.std(axis=0)

iou_summary_df = pd.DataFrame(
    {"IoU": [f"{m:.3f}±{s:.3f}" for m, s in zip(iou_mean, iou_std)]},
    index=["Clase 0", "Clase 1", "Clase 2"]
)

# 4. Global Metrics Summary (Kappa and Agg R2)
metrics_summary_df = pd.DataFrame({
    "Cohen's Kappa": [f"{np.mean(kappa_list):.3f}±{np.std(kappa_list):.3f}"],
    "Agg R2": [f"{np.mean(r2_plot_list):.3f}±{np.std(r2_plot_list):.3f}"],
})


# ==============================================================================
# -------- SAVE WORKBOOK --------
# ==============================================================================
model_name = model.__class__.__name__
sheet_name = f"{model_name}_{sp}"
if todas:
    sheet_name = f"{model_name}_{sp}_todas"

output_path = "./modelos/resultados/Class/ML_cla.xlsx"

# Write first block with 'replace' mode
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    report_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=1)

# Overlay remaining blocks sequentially using dynamic padding lengths
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
    start_iou = len(report_summary_df) + 3
    iou_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=start_iou)
    
    start_metrics = start_iou + len(iou_summary_df) + 3
    metrics_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=start_metrics, index=False)
    
    start_importance = start_metrics + len(metrics_summary_df) + 3
    importance_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=start_importance, index=False)

display(report_summary_df)
display(iou_summary_df)
display(metrics_summary_df)
display(importance_summary_df)

=== Classification Report ===
              precision    recall  f1-score   support
0              0.765420  0.934568  0.841579   810.000
1              0.585366  0.278261  0.377210   345.000
2              0.602305  0.605797  0.604046   345.000
accuracy       0.708000  0.708000  0.708000     0.708
macro avg      0.651030  0.606209  0.607612  1500.000
weighted avg   0.686491  0.708000  0.680141  1500.000
=== Classification Report ===
              precision    recall  f1-score      support
0              0.766600  0.940741  0.844789   810.000000
1              0.598765  0.281159  0.382643   345.000000
2              0.587209  0.585507  0.586357   345.000000
accuracy       0.707333  0.707333  0.707333     0.707333
macro avg      0.650858  0.602469  0.604596  1500.000000
weighted avg   0.686738  0.707333  0.679056  1500.000000
=== Classification Report ===
              precision    recall  f1-score      support
0              0.761044  0.935802  0.839424   810.000000
1              0.57

,precision,recall,f1-score,support
0,0.761±0.004,0.937±0.003,0.840±0.003,810.000±0.000
1,0.582±0.010,0.267±0.010,0.366±0.010,345.000±0.000
2,0.592±0.005,0.590±0.007,0.591±0.006,345.000±0.000
accuracy,0.703±0.003,0.703±0.003,0.703±0.003,0.703±0.003
macro avg,0.645±0.004,0.598±0.004,0.599±0.005,1500.000±0.000
weighted avg,0.681±0.004,0.703±0.003,0.673±0.004,1500.000±0.000


,IoU
Clase 0,0.724±0.004
Clase 1,0.224±0.007
Clase 2,0.419±0.005


,Cohen's Kappa,Agg R2
0,0.473±0.006,0.704±0.031


,Variable,Importance
0,TO,0.199±0.004
1,MLOTST,0.147±0.003
2,SPM,0.128±0.004
3,CHL,0.121±0.003
4,CDM,0.119±0.001
5,VGO,0.116±0.002
6,UGO,0.115±0.003
7,MSEN,0.028±0.001
8,MCOS,0.028±0.001


In [7]:
iters = 10

# -------- Initialize Storage DataFrames/Lists for Iterations --------
importance_all_df = pd.DataFrame(columns=vars_names)
reports_list = []
iou_list = []
kappa_list = []
r2_plot_list = []

for i in range(iters):
    # Set dynamic random state per iteration just like the regression loop
    model = ExtraTreesClassifier(n_estimators=200, max_features="sqrt", n_jobs=-1, random_state=random.randint(0, 1000))
    model.fit(X_train_ml, y_train_ml)

    y_pred_flat = model.predict(X_test_ml)

    # Compute metrics
    report, iou_per_class, kappa, cm, y_true_plot, y_pred_plot, r2_plot = compute_metrics(y_test_ml, y_pred_flat, y_true, m_true)
    
    # Store scalar values
    kappa_list.append(kappa)
    r2_plot_list.append(r2_plot)
    
    # Store dataframes/arrays for structural averaging
    reports_list.append(report)
    iou_list.append(np.array(iou_per_class).flatten())

    # -------- Feature Importance Extraction --------
    raw_importances = model.feature_importances_
    T = X_train.shape[1]   # window (12)
    C = X_train.shape[2]   # number of variables/channels
    H_x = X_train.shape[3] # height (lat)
    W_x = X_train.shape[4] # width (lon)

    reshaped_importances = raw_importances.reshape(T, C, H_x, W_x)
    variable_importances = reshaped_importances.sum(axis=(0, 2, 3))
    
    importance_all_df.loc[len(importance_all_df)] = variable_importances


# ==============================================================================
# -------- POST-LOOP AGGREGATION & FORMATTING (Mean ± Std) --------
# ==============================================================================

# 1. Feature Importance Summary
importance_summary_df = pd.DataFrame({
    "Variable": vars_names, 
    "Importance": [f"{importance_all_df[var].mean():.3f}±{importance_all_df[var].std():.3f}" for var in vars_names],
    "Mean Importance": [importance_all_df[var].mean() for var in vars_names]
})
importance_summary_df = importance_summary_df.sort_values(by="Mean Importance", ascending=False).reset_index(drop=True)
importance_summary_df = importance_summary_df.drop(columns=["Mean Importance"])

# 2. Classification Report Summary (Averages the structural metrics table)
concat_reports = pd.concat(reports_list)
report_mean = concat_reports.groupby(concat_reports.index).mean()
report_std = concat_reports.groupby(concat_reports.index).std()

report_summary_df = pd.DataFrame(index=report_mean.index)
for col in report_mean.columns:
    report_summary_df[col] = [f"{m:.3f}±{s:.3f}" for m, s in zip(report_mean[col], report_std[col])]

# Reorder index to match the original structure of report if necessary
report_summary_df = report_summary_df.reindex(report.index)

# 3. IoU Per Class Summary
iou_arr = np.array(iou_list) # Shape: (iters, 3)
iou_mean = iou_arr.mean(axis=0)
iou_std = iou_arr.std(axis=0)

iou_summary_df = pd.DataFrame(
    {"IoU": [f"{m:.3f}±{s:.3f}" for m, s in zip(iou_mean, iou_std)]},
    index=["Clase 0", "Clase 1", "Clase 2"]
)

# 4. Global Metrics Summary (Kappa and Agg R2)
metrics_summary_df = pd.DataFrame({
    "Cohen's Kappa": [f"{np.mean(kappa_list):.3f}±{np.std(kappa_list):.3f}"],
    "Agg R2": [f"{np.mean(r2_plot_list):.3f}±{np.std(r2_plot_list):.3f}"],
})


# ==============================================================================
# -------- SAVE WORKBOOK --------
# ==============================================================================
model_name = model.__class__.__name__
sheet_name = f"{model_name}_{sp}"
if todas:
    sheet_name = f"{model_name}_{sp}_todas"

output_path = "./modelos/resultados/Class/ML_cla.xlsx"

# Write first block with 'replace' mode
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    report_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=1)

# Overlay remaining blocks sequentially using dynamic padding lengths
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
    start_iou = len(report_summary_df) + 3
    iou_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=start_iou)
    
    start_metrics = start_iou + len(iou_summary_df) + 3
    metrics_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=start_metrics, index=False)
    
    start_importance = start_metrics + len(metrics_summary_df) + 3
    importance_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=start_importance, index=False)

display(report_summary_df)
display(iou_summary_df)
display(metrics_summary_df)
display(importance_summary_df)


=== Classification Report ===
              precision    recall  f1-score      support
0              0.755511  0.930864  0.834071   810.000000
1              0.562874  0.272464  0.367188   345.000000
2              0.591045  0.573913  0.582353   345.000000
accuracy       0.697333  0.697333  0.697333     0.697333
macro avg      0.636477  0.592414  0.594537  1500.000000
weighted avg   0.673377  0.697333  0.668793  1500.000000
=== Classification Report ===
              precision    recall  f1-score  support
0              0.760364  0.928395  0.836020    810.0
1              0.555556  0.275362  0.368217    345.0
2              0.597059  0.588406  0.592701    345.0
accuracy       0.700000  0.700000  0.700000      0.7
macro avg      0.637659  0.597388  0.598979   1500.0
weighted avg   0.675698  0.700000  0.672462   1500.0
=== Classification Report ===
              precision    recall  f1-score      support
0              0.762677  0.928395  0.837416   810.000000
1              0.524096  0

,precision,recall,f1-score,support
0,0.757±0.004,0.929±0.001,0.834±0.003,810.000±0.000
1,0.549±0.019,0.256±0.011,0.349±0.012,345.000±0.000
2,0.590±0.006,0.590±0.010,0.590±0.007,345.000±0.000
accuracy,0.696±0.003,0.696±0.003,0.696±0.003,0.696±0.003
macro avg,0.632±0.006,0.592±0.004,0.591±0.004,1500.000±0.000
weighted avg,0.671±0.004,0.696±0.003,0.667±0.003,1500.000±0.000


,IoU
Clase 0,0.716±0.004
Clase 1,0.211±0.008
Clase 2,0.419±0.006


,Cohen's Kappa,Agg R2
0,0.462±0.005,0.694±0.016


,Variable,Importance
0,TO,0.158±0.003
1,SPM,0.119±0.002
2,MLOTST,0.118±0.003
3,CDM,0.117±0.002
4,CHL,0.115±0.002
5,VGO,0.114±0.001
6,UGO,0.112±0.002
7,MSEN,0.075±0.004
8,MCOS,0.073±0.005
